# Allen–Cahn PDE conditioning of GP regression

This notebook fits a Gaussian process (GP) to sparse, early-time observations of a solution to the Allen–Cahn equation

$$u_t = \epsilon\, u_{xx} + 5\,(u - u^3), \qquad x \in [-1, 1],\ t \in [0, 1],\ \epsilon = 10^{-4},$$

and then uses guided flow sampling to draw space-time fields from the GP posterior that also approximately satisfy the PDE with periodic boundary conditions. The data only cover $t < 0.28$, so at later times the samples are shaped by the physics alone, without any extra observations.

## 1. Setup

The fields live on an $H \times W$ grid (space × time). `NUM_TRAIN` observations are drawn from times before `T_TRAIN_MAX`, and `SEED` fixes both the choice of training points and the conditioned samples.

In [ ]:
import sys
import time
from pathlib import Path

import gpytorch
import numpy as np
import pandas as pd
import torch

from flow_inference.conditions import EqualityCondition
from flow_inference.flow_utils import build_grid_in_box
from flow_inference.flows import GaussianProcessFlow
from flow_inference.gp_utils import fit_kernel_hyperparameters, gp_posterior
from flow_inference.guidance import GuidanceMC, GuidancePiGDM
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # make examples/plotting.py importable
from plotting import show_fields

In [ ]:
H = 50                # spatial grid points
W = 20                # temporal grid points
NOISE = 0.01**2       # observation noise variance
NUM_TRAIN = 256       # training observations
T_TRAIN_MAX = 0.28    # observations are drawn from t < T_TRAIN_MAX
NUM_SAMPLES = 5
NUM_STEPS = 1_000
SEED = 1234

device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Data and GP posterior

### 2.1 Data

The training set is `NUM_TRAIN` points drawn at random from the early-time part of a reference solution. Space is rescaled from $[-1, 1]$ to $[0, 1]$, so both inputs live in the unit box used by the sampling grid. The grid's $H$ points $x_i = -1 + 2i/H$ stop one step short of $x = 1$, which is the periodic image of $x = -1$.

In [ ]:
data_df = pd.read_csv("../data/allen_cahn.csv")
X_all = data_df[["x", "t"]].to_numpy()
X_all[:, 0] = 0.5 * X_all[:, 0] + 0.5  # rescale x to [0, 1]
y_all = data_df["y"].to_numpy()

np.random.seed(SEED)
early = X_all[:, 1] < T_TRAIN_MAX
idx = np.random.choice(np.arange(early.sum()), NUM_TRAIN)
X_train = torch.tensor(X_all[early][idx], device=device)
y_train = torch.tensor(y_all[early][idx], device=device)

### 2.2 GP posterior

The hyperparameters of an anisotropic RBF kernel (separate space and time lengthscales) are fitted by type-II maximum likelihood, starting from the values set below. `gp_posterior` then forms the conjugate posterior mean and kernel.

In [ ]:
coords = build_grid_in_box(H, W, device)  # (H*W, 2)

kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel(ard_num_dims=2)).to(device)
kernel.base_kernel.lengthscale = torch.tensor([0.05, 0.1], device=device)  # initial (space, time)
kernel.outputscale = 1.0
kernel = fit_kernel_hyperparameters(
    X_train=X_train, y_train=y_train, kernel=kernel, noise=NOISE, device=device)
posterior_mean, posterior_kernel = gp_posterior(kernel, X_train, y_train, noise=NOISE)

### 2.3 Prior samples

A `GaussianProcessFlow` on the $H \times W$ grid starts from this posterior. Unguided samples are draws from the data-only posterior: each panel is one field, with space vertically and time horizontally.

In [ ]:
flow = GaussianProcessFlow(posterior_kernel, posterior_mean, coords, output_shape=(H, W), device=device)
sampler = IntegratorSampler(flow=flow)

samples = sampler.sample(NUM_SAMPLES, guidance=None, num_steps=NUM_STEPS)
show_fields(samples, cmap="bwr")

## 3. Allen–Cahn PDE condition

The PDE is imposed as an `EqualityCondition` on the finite-difference residual

$$r = u_t - \epsilon u_{xx} + 5u^3 - 5u,$$

which should be zero. $u_t$ is a central difference, so the residual is evaluated at every interior time. $u_{xx}$ uses a stencil that wraps around in $x$, which imposes the periodic boundary conditions exactly, with no separate boundary term. `as_grid` views the flat field as $(B, C, H, W)$, taking the grid shape from the flow once the condition is passed to the guidance.

The residual is evaluated at every interior grid cell, so the condition passes the cell area as quadrature `weights`. The log-likelihood is then

$$-\frac{1}{2\sigma^2}\sum_{ij} r_{ij}^2\,\Delta x\,\Delta t \;\approx\; -\frac{1}{2\sigma^2}\int r(x, t)^2\,dx\,dt,$$

so $\sigma$ is a tolerance on the root-mean-square residual over the whole domain. Without the weights every extra grid cell would add another penalty term, and refining $H$ or $W$ would enforce the PDE more tightly; with them, $\sigma$ means the same thing at any resolution. The value used below, $\sigma = 4.5\times10^{-6}$, corresponds to a tolerance of $10^{-4}$ per cell on this $50 \times 20$ grid.

In [ ]:
class AllenCahnCondition(EqualityCondition):
    def __init__(self, sigma, weights, eps=1e-4):
        super().__init__(sigma=sigma, weights=weights)  # weights: the cell area dx * dt
        self.register_buffer("eps", torch.tensor(float(eps)))

    def residual(self, x):  # (B, 1, H*W) -> (B, 1, H*(W-2))
        u = self.as_grid(x)[:, 0]  # (B, H, W)
        B, H, W = u.shape
        dt = 1.0 / W
        dx = 2.0 / H  # periodic grid: x_i = -1 + i*dx, and x = 1 wraps to x = -1

        u_t = (u[:, :, 2:] - u[:, :, :-2]) / (2.0 * dt)  # central difference, interior times
        u_xx = (u.roll(-1, dims=1) - 2.0 * u + u.roll(1, dims=1)) / dx**2  # periodic in x
        u_int = u[:, :, 1:-1]
        residual = u_t - self.eps * u_xx[:, :, 1:-1] + 5.0 * (u_int.pow(3) - u_int)
        return residual.reshape(B, 1, -1)


DX, DT = 2.0 / H, 1.0 / W  # the grid cell, as used in the residual's stencils
condition = AllenCahnCondition(sigma=4.5e-6, weights=DX * DT).to(device)

## 4. Conditioned sampling

`GuidanceMC` estimates the guidance term at each step from one Monte Carlo draw of the clean field given the current state. Instead of a fixed number of steps, the sampler here uses an adaptive step-size controller (`adaptive=True`), which takes small steps only where the guided dynamics need them: `atol` and `rtol` are tolerances on the local integration error, and `max_steps` caps the number of trial steps.

In [ ]:
guidance = GuidanceMC(flow, condition, num_mc_samples=1).to(device)

torch.manual_seed(SEED)
samples = sampler.sample(NUM_SAMPLES, guidance=guidance, adaptive=True, atol=1e-3, rtol=1e-3, max_steps=100_000)
show_fields(samples, cmap="bwr")

## 5. Π-GDM guidance

`GuidanceMC` steers each sample by the gradient of the condition at a guess of the clean field, ignoring how uncertain that guess still is. With a tolerance as tight as $\sigma = 4.5\times10^{-6}$ that gradient is huge and changes abruptly, which is why Section 4 needs an adaptive solver taking many small steps.

`GuidancePiGDM` accounts for that uncertainty: it implements Π-GDM (Song et al., 2023), the guidance of the spatial-prediction paper by Zammit-Mangion et al. (2026). At each step the clean latent given the current state $z_t$ is Gaussian, with mean $\mu_t = \alpha_t z_t$ and variance $v_t = 1 - \alpha_t^2$. Linearising the residual $r$ around $\mu_t$ makes the guidance a Gaussian calculation,

$$\nabla \log p(y \mid z_t) \approx -\alpha_t\, J^\top \left(v_t\, J J^\top + \Sigma\right)^{-1} r(\mu_t),$$

where $J$ is the residual's Jacobian and $\Sigma$ the squared tolerances. While $v_t$ is large the pull is gentle, and it sharpens only as the sample settles, so the drift stays bounded however small $\sigma$ is: there is no clipping, and ten fixed steps are enough here. `num_iterations=3` moves the linearisation point towards the most likely clean field with two further Gauss–Newton steps, which helps with the nonlinear $u^3$ term. Each step solves a least-squares problem (by LSMR, using only Jacobian–vector products), so it costs more than a `GuidanceMC` step, but far fewer steps are needed.

Below, both guidances draw the same number of samples from the same seed. Each run is timed and scored by the root-mean-square PDE residual.

In [ ]:
def timed_sample(sampler, guidance, num_samples, **kwargs):
    """Guided samples, and the wall time taken to draw them."""
    torch.manual_seed(SEED)
    if device == "cuda":
        torch.cuda.synchronize()  # time the work, not the asynchronous launch
    start = time.perf_counter()
    out = sampler.sample(num_samples, guidance=guidance, verbose=False, **kwargs)
    if device == "cuda":
        torch.cuda.synchronize()
    return out, time.perf_counter() - start


def rms_residual(fields):
    """Root-mean-square Allen–Cahn residual over the samples."""
    return condition.residual(fields.reshape(fields.shape[0], 1, -1)).pow(2).mean().sqrt().item()


mc_samples, mc_time = timed_sample(
    sampler, guidance, NUM_SAMPLES, adaptive=True, atol=1e-3, rtol=1e-3, max_steps=100_000)
mc_steps = sampler.adaptive_stats["accepted"] + sampler.adaptive_stats["rejected"]

pigdm_guidance = GuidancePiGDM(flow, condition, num_iterations=3).to(device)
pigdm_samples, pigdm_time = timed_sample(sampler, pigdm_guidance, NUM_SAMPLES, num_steps=10)

print(f"GuidanceMC,    adaptive ({mc_steps} trial steps): {mc_time:5.2f}s   RMS residual {rms_residual(mc_samples):.3f}")
print(f"GuidancePiGDM, 10 steps:                   {pigdm_time:5.2f}s   RMS residual {rms_residual(pigdm_samples):.3f}")
show_fields(pigdm_samples, cmap="bwr")

## 6. Summary

* The Allen–Cahn equation enters as an `EqualityCondition` on its finite-difference residual, with periodic boundaries handled by a wrap-around stencil rather than a separate condition.
* Weighting the residual by the cell area makes $\sigma$ a resolution-independent tolerance on the root-mean-square residual.
* `GuidancePiGDM` (Section 5) accounts for the uncertainty of the clean field, so its drift stays bounded under a tight tolerance: 10 fixed steps give a lower residual than the adaptive `GuidanceMC` run, in less time.
* Compare Section 4 with the prior samples in Section 2.3, particularly at later times (the right of each panel), where there are no observations: there the physics, not the data, shapes the conditioned fields.